In [2]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


In [3]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [4]:
DATA_PATH = "Loan_default.csv"
OUT_DIR = "output"
PLOT_DIR = os.path.join(OUT_DIR, "eda_plots")
TARGET = "Default"
RANDOM_STATE = 42
TEST_SIZE = 0.20
SHOW_PLOTS = False

In [5]:
os.makedirs(PLOT_DIR, exist_ok=True)


def save_plot(name):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"{name}.png"), dpi=120)
    if SHOW_PLOTS:
        plt.show()
    plt.close()

In [6]:
df = pd.read_csv(DATA_PATH)


print("1. DATA OVERVIEW")

print("Shape:", df.shape)
print("\nFirst rows:\n", df.head())
print("\nInfo:")
df.info()
print("\nNumeric summary:\n", df.describe().T)
print("\nCategorical summary:\n", df.describe(include="object").T)

1. DATA OVERVIEW
Shape: (255347, 18)

First rows:
        LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio    Education  \
0               4         15.23        36      0.44   Bachelor's   
1               1          4.81        60      0.68     Master's   
2               3         21.17        24      0.31     Master's   
3               3          7.07        24      0.23  High School   
4               4          6.51        48      0.73   Bachelor's   

  EmploymentType MaritalStatus HasMortgage HasDependents LoanPurpose  \
0      Full-time      Divorced         Ye

In [7]:
missing = df.isna().sum()
print("Missing values per column:\n", missing[missing > 0] if missing.any() else "None")

Missing values per column:
 None


In [8]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate LoanIDs:", df["LoanID"].duplicated().sum())
df = df.drop_duplicates().drop_duplicates(subset="LoanID")

Duplicate rows: 0
Duplicate LoanIDs: 0


In [9]:
cat_cols_raw = df.select_dtypes(include="object").columns.drop("LoanID")
for c in cat_cols_raw:
    df[c] = df[c].str.strip()

In [10]:
checks = {
    "Age must be 18-100": ~df["Age"].between(18, 100),
    "Income must be > 0": df["Income"] <= 0,
    "LoanAmount must be > 0": df["LoanAmount"] <= 0,
    "CreditScore must be 300-850": ~df["CreditScore"].between(300, 850),
    "MonthsEmployed must be >= 0": df["MonthsEmployed"] < 0,
    "InterestRate must be > 0": df["InterestRate"] <= 0,
    "DTIRatio must be 0-1": ~df["DTIRatio"].between(0, 1),
    "Target must be 0/1": ~df[TARGET].isin([0, 1]),
}
for rule, bad in checks.items():
    print(f"{rule}: {bad.sum()} invalid rows")
df = df[~pd.concat(checks.values(), axis=1).any(axis=1)]

Age must be 18-100: 0 invalid rows
Income must be > 0: 0 invalid rows
LoanAmount must be > 0: 0 invalid rows
CreditScore must be 300-850: 0 invalid rows
MonthsEmployed must be >= 0: 0 invalid rows
InterestRate must be > 0: 0 invalid rows
DTIRatio must be 0-1: 0 invalid rows
Target must be 0/1: 0 invalid rows


In [11]:
num_cols_raw = [c for c in df.select_dtypes(include="number").columns if c != TARGET]
outlier_report = {}
for c in num_cols_raw:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_report[c] = int(((df[c] < lo) | (df[c] > hi)).sum())
    # df[c] = df[c].clip(lo, hi)   # <- optional winsorizing
print("\nIQR outlier counts:", outlier_report)


IQR outlier counts: {'Age': 0, 'Income': 0, 'LoanAmount': 0, 'CreditScore': 0, 'MonthsEmployed': 0, 'NumCreditLines': 0, 'InterestRate': 0, 'LoanTerm': 0, 'DTIRatio': 0}


In [12]:
for c in cat_cols_raw:
    df[c] = df[c].astype("category")

df = df.reset_index(drop=True)
print("Shape after cleaning:", df.shape)

Shape after cleaning: (255347, 18)


In [13]:
cat_cols = [c for c in df.select_dtypes(include="category").columns if c != "LoanID"]
num_cols = [c for c in df.select_dtypes(include="number").columns if c != TARGET]

In [14]:
counts = df[TARGET].value_counts()
print("Target counts:\n", counts)
print("Default rate: {:.2%}".format(df[TARGET].mean()))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=TARGET, data=df, ax=ax)
ax.set_title("Target distribution (0 = Paid, 1 = Default)")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
save_plot("01_target_distribution")

Target counts:
 Default
0    225694
1     29653
Name: count, dtype: int64
Default rate: 11.61%


In [15]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(df[c], bins=40, kde=True, ax=ax)
    ax.set_title(c)
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
save_plot("02_numeric_distributions")

In [16]:
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [17]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(x=TARGET, y=c, data=df, ax=ax)
    ax.set_title(f"{c} by {TARGET}")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
save_plot("03_numeric_vs_target")

In [18]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, c in zip(axes.ravel(), cat_cols):
    rate = df.groupby(c, observed=True)[TARGET].mean().sort_values()
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax)
    ax.axhline(df[TARGET].mean(), color="red", ls="--", lw=1, label="overall")
    ax.set_title(f"Default rate by {c}")
    ax.set_ylabel("Default rate")
    ax.tick_params(axis="x", rotation=25)
for ax in axes.ravel()[len(cat_cols):]:
    ax.axis("off")
save_plot("04_categorical_vs_default_rate")

In [19]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), ["Age", "Income", "CreditScore",
                                 "InterestRate", "MonthsEmployed", "LoanAmount"]):
    binned = pd.qcut(df[c], 10, duplicates="drop")
    rate = df.groupby(binned, observed=True)[TARGET].mean()
    ax.plot(range(len(rate)), rate.values, marker="o")
    ax.set_title(f"Default rate by {c} decile")
    ax.set_xlabel("Decile (low -> high)")
    ax.set_ylabel("Default rate")
save_plot("05_default_rate_by_decile")

In [20]:
corr = df[num_cols + [TARGET]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
save_plot("06_correlation_heatmap")
print("\nCorrelation with target:\n", corr[TARGET].drop(TARGET).sort_values())


Correlation with target:
 Age              -0.167783
Income           -0.099119
MonthsEmployed   -0.097374
CreditScore      -0.034166
LoanTerm          0.000545
DTIRatio          0.019236
NumCreditLines    0.028330
LoanAmount        0.086659
InterestRate      0.131273
Name: Default, dtype: float64


In [21]:
fe = df.copy()

In [22]:
fe["LoanToIncome"] = fe["LoanAmount"] / fe["Income"]
fe["Monthly_Income"] = fe["Income"] / 12

In [23]:
r = fe["InterestRate"] / 100 / 12                       # monthly rate
n = fe["LoanTerm"]                                      # months
fe["MonthlyPayment"] = fe["LoanAmount"] * r / (1 - (1 + r) ** (-n))
fe["PaymentToIncome"] = fe["MonthlyPayment"] / fe["Monthly_Income"]
fe["TotalInterestCost"] = fe["MonthlyPayment"] * n - fe["LoanAmount"]

In [24]:
fe["DebtBurden"] = fe["DTIRatio"] * fe["LoanAmount"]
fe["IncomePerCreditLine"] = fe["Income"] / fe["NumCreditLines"]
fe["Rate_x_Term"] = fe["InterestRate"] * fe["LoanTerm"]
fe["CreditScore_x_Income"] = fe["CreditScore"] * fe["Income"]

In [25]:
fe["EmploymentYears"] = fe["MonthsEmployed"] / 12
fe["EmploymentToAge"] = fe["EmploymentYears"] / fe["Age"]
fe["IsUnstableJob"] = fe["EmploymentType"].isin(["Unemployed", "Part-time"]).astype(int)

In [26]:
fe["AgeGroup"] = pd.cut(fe["Age"], bins=[17, 25, 35, 45, 55, 70],
                        labels=["18-25", "26-35", "36-45", "46-55", "56+"])
fe["CreditBand"] = pd.cut(fe["CreditScore"], bins=[299, 579, 669, 739, 799, 850],
                          labels=["Poor", "Fair", "Good", "VeryGood", "Exceptional"])
fe["IncomeBand"] = pd.qcut(fe["Income"], 4, labels=["Low", "MidLow", "MidHigh", "High"])

In [27]:
for c in ["Income", "LoanAmount", "TotalInterestCost"]:
    fe[f"Log_{c}"] = np.log1p(fe[c])

In [28]:
fe = fe.replace([np.inf, -np.inf], np.nan)
print("NaNs created by feature engineering:", int(fe.isna().sum().sum()))
fe = fe.fillna(fe.median(numeric_only=True))

NaNs created by feature engineering: 0


In [29]:
new_feats = [c for c in fe.columns if c not in df.columns]
print("New features:", new_feats)
print("\nCorrelation of new numeric features with target:")
print(fe[[c for c in new_feats if fe[c].dtype != "category"] + [TARGET]]
      .corr()[TARGET].drop(TARGET).sort_values())

New features: ['LoanToIncome', 'Monthly_Income', 'MonthlyPayment', 'PaymentToIncome', 'TotalInterestCost', 'DebtBurden', 'IncomePerCreditLine', 'Rate_x_Term', 'CreditScore_x_Income', 'EmploymentYears', 'EmploymentToAge', 'IsUnstableJob', 'AgeGroup', 'CreditBand', 'IncomeBand', 'Log_Income', 'Log_LoanAmount', 'Log_TotalInterestCost']

Correlation of new numeric features with target:
Log_Income              -0.117825
Monthly_Income          -0.099119
CreditScore_x_Income    -0.098860
EmploymentYears         -0.097374
IncomePerCreditLine     -0.076733
EmploymentToAge          0.003440
IsUnstableJob            0.035799
MonthlyPayment           0.070967
DebtBurden               0.074634
Log_LoanAmount           0.077109
Rate_x_Term              0.090495
TotalInterestCost        0.119179
Log_TotalInterestCost    0.119984
PaymentToIncome          0.151861
LoanToIncome             0.178963
Name: Default, dtype: float64


In [30]:
X = fe.drop(columns=[TARGET, "LoanID"])
y = fe[TARGET]

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

In [32]:
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train default rate: {:.3%} | Test default rate: {:.3%}".format(
    y_train.mean(), y_test.mean()))


Train: (204277, 34) | Test: (51070, 34)
Train default rate: 11.613% | Test default rate: 11.613%


In [33]:
binary_cols = ["HasMortgage", "HasDependents", "HasCoSigner"]            # Yes / No
ordinal_cols = {                                                         # natural order
    "Education": ["High School", "Bachelor's", "Master's", "PhD"],
    "AgeGroup": ["18-25", "26-35", "36-45", "46-55", "56+"],
    "CreditBand": ["Poor", "Fair", "Good", "VeryGood", "Exceptional"],
    "IncomeBand": ["Low", "MidLow", "MidHigh", "High"],
}
nominal_cols = ["EmploymentType", "MaritalStatus", "LoanPurpose"]        # no order
numeric_features = [c for c in X_train.columns
                    if c not in binary_cols + list(ordinal_cols) + nominal_cols
                    and c != "IsUnstableJob"]
passthrough_cols = ["IsUnstableJob"]

In [34]:
for col in binary_cols + list(ordinal_cols) + nominal_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("bin", OrdinalEncoder(categories=[["No", "Yes"]] * len(binary_cols)), binary_cols),
        ("ord", OrdinalEncoder(categories=list(ordinal_cols.values())),
         list(ordinal_cols.keys())),
        ("nom", OneHotEncoder(drop="first", handle_unknown="ignore",
                              sparse_output=False), nominal_cols),
        ("pass", "passthrough", passthrough_cols),
    ],
    verbose_feature_names_out=False,
)

In [35]:
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_proc = pd.DataFrame(X_train_proc, columns=feature_names, index=X_train.index)
X_test_proc = pd.DataFrame(X_test_proc, columns=feature_names, index=X_test.index)

print("Processed train shape:", X_train_proc.shape)
print("Processed test shape :", X_test_proc.shape)
print("Any NaN left?        :", X_train_proc.isna().any().any(), X_test_proc.isna().any().any())
print("Final features:", list(feature_names))

Processed train shape: (204277, 40)
Processed test shape : (51070, 40)
Any NaN left?        : False False
Final features: ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'LoanToIncome', 'Monthly_Income', 'MonthlyPayment', 'PaymentToIncome', 'TotalInterestCost', 'DebtBurden', 'IncomePerCreditLine', 'Rate_x_Term', 'CreditScore_x_Income', 'EmploymentYears', 'EmploymentToAge', 'Log_Income', 'Log_LoanAmount', 'Log_TotalInterestCost', 'HasMortgage', 'HasDependents', 'HasCoSigner', 'Education', 'AgeGroup', 'CreditBand', 'IncomeBand', 'EmploymentType_Part-time', 'EmploymentType_Self-employed', 'EmploymentType_Unemployed', 'MaritalStatus_Married', 'MaritalStatus_Single', 'LoanPurpose_Business', 'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other', 'IsUnstableJob']


In [36]:
print("\nScaled train mean/std (first 5 numeric):")
print(X_train_proc[numeric_features[:5]].agg(["mean", "std"]).round(3))


Scaled train mean/std (first 5 numeric):
      Age  Income  LoanAmount  CreditScore  MonthsEmployed
mean -0.0     0.0        -0.0          0.0            -0.0
std   1.0     1.0         1.0          1.0             1.0


In [38]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Negatives: {neg}, Positives: {pos}, scale_pos_weight = {scale_pos_weight:.2f}")
print("""Options (pick one, apply to TRAIN only):
  - class_weight='balanced'          (LogisticRegression, RandomForest)
  - scale_pos_weight=<value above>   (XGBoost)
  - SMOTE / RandomUnderSampler       (pip install imbalanced-learn)
  - Threshold tuning on predicted probabilities (recommended; use PR-AUC)""")

Negatives: 180555, Positives: 23722, scale_pos_weight = 7.61
Options (pick one, apply to TRAIN only):
  - class_weight='balanced'          (LogisticRegression, RandomForest)
  - scale_pos_weight=<value above>   (XGBoost)
  - SMOTE / RandomUnderSampler       (pip install imbalanced-learn)
  - Threshold tuning on predicted probabilities (recommended; use PR-AUC)


In [39]:
X_train_proc.assign(**{TARGET: y_train.values}).to_csv(
    os.path.join(OUT_DIR, "train_processed.csv"), index=False)
X_test_proc.assign(**{TARGET: y_test.values}).to_csv(
    os.path.join(OUT_DIR, "test_processed.csv"), index=False)

import joblib
joblib.dump(preprocessor, os.path.join(OUT_DIR, "preprocessor.joblib"))

print("\nSaved to ./%s: train_processed.csv, test_processed.csv, preprocessor.joblib" % OUT_DIR)
print("EDA plots saved to ./%s" % PLOT_DIR)


Saved to ./output: train_processed.csv, test_processed.csv, preprocessor.joblib
EDA plots saved to ./output\eda_plots


In [43]:
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from xgboost import XGBClassifier

In [44]:

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


In [45]:
OUT_DIR = "output"
PLOT_DIR = os.path.join(OUT_DIR, "model_plots")
TARGET = "Default"
RANDOM_STATE = 42
CV_FOLDS = 5
CV_SAMPLE = 50000   # rows used for cross-validation (speed). Set to None to use ALL train rows.
os.makedirs(PLOT_DIR, exist_ok=True)


def save_plot(name):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"{name}.png"), dpi=120)
    plt.close()

In [46]:
train = pd.read_csv(os.path.join(OUT_DIR, "train_processed.csv"))
test = pd.read_csv(os.path.join(OUT_DIR, "test_processed.csv"))

X_train, y_train = train.drop(columns=TARGET), train[TARGET]
X_test, y_test = test.drop(columns=TARGET), test[TARGET]
print("Train:", X_train.shape, "| Test:", X_test.shape)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos  # scale_pos_weight for XGBoost

Train: (204277, 40) | Test: (51070, 40)


In [47]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=20,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=300, max_leaf_nodes=31,
        class_weight="balanced", early_stopping=True, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr",
        n_jobs=-1, random_state=RANDOM_STATE),
}

In [48]:
print("\n" + "=" * 70)
print(f"2. {CV_FOLDS}-FOLD CROSS-VALIDATION (train set)")
print("=" * 70)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
if CV_SAMPLE and CV_SAMPLE < len(X_train):
    X_cv, _, y_cv, _ = train_test_split(X_train, y_train, train_size=CV_SAMPLE,
                                        stratify=y_train, random_state=RANDOM_STATE)
    print(f"(CV on a stratified {CV_SAMPLE:,}-row sample for speed; set CV_SAMPLE=None for all rows)")
else:
    X_cv, y_cv = X_train, y_train
cv_rows = []
for name, model in models.items():
    print(f"  running CV for {name} ...", flush=True)
    res = cross_validate(model, X_cv, y_cv, cv=cv,
                         scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
                         n_jobs=1)
    cv_rows.append({
        "Model": name,
        "CV ROC-AUC": res["test_roc_auc"].mean(),
        "CV ROC-AUC std": res["test_roc_auc"].std(),
        "CV PR-AUC": res["test_pr_auc"].mean(),
        "CV PR-AUC std": res["test_pr_auc"].std(),
    })
    print(f"{name:22s} ROC-AUC {res['test_roc_auc'].mean():.4f} "
          f"(+/-{res['test_roc_auc'].std():.4f}) | PR-AUC {res['test_pr_auc'].mean():.4f}")
cv_df = pd.DataFrame(cv_rows).sort_values("CV PR-AUC", ascending=False)
cv_df.to_csv(os.path.join(OUT_DIR, "cv_results.csv"), index=False)


2. 5-FOLD CROSS-VALIDATION (train set)
(CV on a stratified 50,000-row sample for speed; set CV_SAMPLE=None for all rows)
  running CV for LogisticRegression ...
LogisticRegression     ROC-AUC 0.7548 (+/-0.0018) | PR-AUC 0.3257
  running CV for RandomForest ...
RandomForest           ROC-AUC 0.7395 (+/-0.0029) | PR-AUC 0.3047
  running CV for HistGradientBoosting ...
HistGradientBoosting   ROC-AUC 0.7428 (+/-0.0026) | PR-AUC 0.3084
  running CV for XGBoost ...
XGBoost                ROC-AUC 0.7428 (+/-0.0026) | PR-AUC 0.3107


In [49]:
fitted, test_proba, rows = {}, {}, []
for name, model in models.items():
    print(f"  fitting {name} on full train ...", flush=True)
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    fitted[name], test_proba[name] = model, p
    rows.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, p),
        "PR-AUC": average_precision_score(y_test, p),
        "Precision@0.5": precision_score(y_test, pred),
        "Recall@0.5": recall_score(y_test, pred),
        "F1@0.5": f1_score(y_test, pred),
    })
results = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
print(results.round(4).to_string(index=False))
results.to_csv(os.path.join(OUT_DIR, "test_results.csv"), index=False)

baseline_pr = y_test.mean()
print(f"\nNo-skill PR-AUC baseline = default rate = {baseline_pr:.3f}")

  fitting LogisticRegression on full train ...
  fitting RandomForest on full train ...
  fitting HistGradientBoosting on full train ...
  fitting XGBoost on full train ...
               Model  ROC-AUC  PR-AUC  Precision@0.5  Recall@0.5  F1@0.5
  LogisticRegression   0.7613  0.3361         0.2279      0.6970  0.3435
             XGBoost   0.7589  0.3333         0.2300      0.6861  0.3446
HistGradientBoosting   0.7579  0.3316         0.2297      0.6761  0.3429
        RandomForest   0.7496  0.3180         0.2571      0.5763  0.3555

No-skill PR-AUC baseline = default rate = 0.116


In [50]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, p in test_proba.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rc, pr, label=f"{name} ({average_precision_score(y_test, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(title="ROC curves", xlabel="False positive rate", ylabel="True positive rate")
axes[1].axhline(baseline_pr, color="k", ls="--", lw=1, label="No skill")
axes[1].set(title="Precision-Recall curves", xlabel="Recall", ylabel="Precision")
axes[0].legend(); axes[1].legend()
save_plot("01_roc_pr_curves")

best_name = results.loc[0, "Model"]
print("Best model by test PR-AUC:", best_name)

Best model by test PR-AUC: LogisticRegression


In [51]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.20, stratify=y_train, random_state=RANDOM_STATE)

from sklearn.base import clone
tuned = clone(models[best_name]).fit(X_fit, y_fit)
val_p = tuned.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
val_f1 = [f1_score(y_val, val_p >= t) for t in thresholds]
best_t_f1 = thresholds[int(np.argmax(val_f1))]

# Business-style rule: highest threshold that still reaches >= 60% recall
TARGET_RECALL = 0.60
ok = [t for t in thresholds if recall_score(y_val, val_p >= t) >= TARGET_RECALL]
best_t_recall = max(ok) if ok else best_t_f1

# Cost-based rule: a missed default (FN) costs more than a rejected good loan (FP)
COST_FN, COST_FP = 5, 1


In [52]:
def total_cost(y_true, p, t):
    pred = p >= t
    fn = ((y_true == 1) & (~pred)).sum()
    fp = ((y_true == 0) & pred).sum()
    return COST_FN * fn + COST_FP * fp


val_cost = [total_cost(y_val, val_p, t) for t in thresholds]
best_t_cost = thresholds[int(np.argmin(val_cost))]

print(f"Best threshold (max F1 on validation)     : {best_t_f1:.2f}")
print(f"Threshold for >= {TARGET_RECALL:.0%} recall (validation)    : {best_t_recall:.2f}")
print(f"Best threshold (cost FN={COST_FN}, FP={COST_FP}, validation): {best_t_cost:.2f}")


Best threshold (max F1 on validation)     : 0.63
Threshold for >= 60% recall (validation)    : 0.55
Best threshold (cost FN=5, FP=1, validation): 0.63


In [53]:
final_p = test_proba[best_name]
print(f"\nTest-set metrics for {best_name}:")
print(f"{'Strategy':28s}{'Thr':>6s}{'Prec':>8s}{'Recall':>8s}{'F1':>8s}{'Cost':>10s}")
for label, t in [("Default 0.50", 0.50),
                 ("Max F1", best_t_f1),
                 (f">= {TARGET_RECALL:.0%} recall", best_t_recall),
                 ("Min cost", best_t_cost)]:
    pred = (final_p >= t).astype(int)
    print(f"{label:28s}{t:6.2f}{precision_score(y_test, pred):8.3f}"
          f"{recall_score(y_test, pred):8.3f}{f1_score(y_test, pred):8.3f}"
          f"{total_cost(y_test, final_p, t):10d}")

FINAL_THRESHOLD = best_t_f1  


Test-set metrics for LogisticRegression:
Strategy                       Thr    Prec  Recall      F1      Cost
Default 0.50                  0.50   0.228   0.697   0.343     22992
Max F1                        0.63   0.303   0.496   0.376     21722
>= 60% recall                 0.55   0.253   0.624   0.360     22066
Min cost                      0.63   0.303   0.496   0.376     21722


In [54]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
prec = [precision_score(y_val, val_p >= t, zero_division=0) for t in thresholds]
rec = [recall_score(y_val, val_p >= t) for t in thresholds]
axes[0].plot(thresholds, prec, label="Precision")
axes[0].plot(thresholds, rec, label="Recall")
axes[0].plot(thresholds, val_f1, label="F1")
axes[0].axvline(best_t_f1, color="k", ls="--", lw=1, label=f"Max F1 ({best_t_f1:.2f})")
axes[0].set(title="Metrics vs threshold (validation)", xlabel="Threshold")
axes[0].legend()
axes[1].plot(thresholds, val_cost, color="tab:red")
axes[1].axvline(best_t_cost, color="k", ls="--", lw=1)
axes[1].set(title=f"Cost vs threshold (FN={COST_FN}x FP)", xlabel="Threshold", ylabel="Total cost")
save_plot("02_threshold_tuning")

In [55]:
pred_final = (final_p >= FINAL_THRESHOLD).astype(int)
print(f"\nClassification report @ threshold {FINAL_THRESHOLD:.2f}:")
print(classification_report(y_test, pred_final, target_names=["Paid", "Default"]))
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_final, display_labels=["Paid", "Default"],
                                        cmap="Blues", ax=ax)
ax.set_title(f"{best_name} @ {FINAL_THRESHOLD:.2f}")
save_plot("03_confusion_matrix")


Classification report @ threshold 0.63:
              precision    recall  f1-score   support

        Paid       0.93      0.85      0.89     45139
     Default       0.30      0.50      0.38      5931

    accuracy                           0.81     51070
   macro avg       0.62      0.67      0.63     51070
weighted avg       0.86      0.81      0.83     51070



In [56]:
best_model = fitted[best_name]
sample = X_test.sample(min(15000, len(X_test)), random_state=RANDOM_STATE)
perm = permutation_importance(best_model, sample, y_test.loc[sample.index],
                              scoring="average_precision", n_repeats=5,
                              random_state=RANDOM_STATE, n_jobs=1)
imp = (pd.Series(perm.importances_mean, index=X_test.columns)
       .sort_values(ascending=False))
print(imp.head(15).round(4))

plt.figure(figsize=(8, 7))
imp.head(15).iloc[::-1].plot(kind="barh")
plt.title(f"Top 15 features - {best_name} (permutation, PR-AUC drop)")
plt.xlabel("Mean decrease in PR-AUC")
save_plot("04_feature_importance")

LoanToIncome                    0.1105
Age                             0.0905
InterestRate                    0.0562
MonthsEmployed                  0.0105
EmploymentYears                 0.0105
IsUnstableJob                   0.0074
HasCoSigner                     0.0067
EmploymentType_Unemployed       0.0050
MaritalStatus_Married           0.0048
HasDependents                   0.0043
Log_Income                      0.0034
Education                       0.0033
NumCreditLines                  0.0030
EmploymentType_Self-employed    0.0029
CreditScore                     0.0025
dtype: float64


In [57]:
lr = fitted["LogisticRegression"]
coef = pd.Series(lr.coef_[0], index=X_train.columns).sort_values()
print("\nLogReg strongest coefficients (+ = raises default risk):")
print(pd.concat([coef.head(5), coef.tail(5)]).round(3))


LogReg strongest coefficients (+ = raises default risk):
Age                            -0.563
HasCoSigner                    -0.265
HasDependents                  -0.258
MaritalStatus_Married          -0.203
LoanPurpose_Home               -0.190
EmploymentType_Unemployed       0.201
IsUnstableJob                   0.237
EmploymentType_Self-employed    0.254
InterestRate                    0.440
LoanToIncome                    0.445
dtype: float64


In [58]:
frac_pos, mean_pred = calibration_curve(y_test, final_p, n_bins=10, strategy="quantile")
plt.figure(figsize=(5.5, 5))
plt.plot(mean_pred, frac_pos, marker="o", label=best_name)
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed default rate")
plt.title("Calibration curve")
plt.legend()
save_plot("05_calibration")

In [59]:
joblib.dump({"model": best_model, "threshold": float(FINAL_THRESHOLD),
             "features": list(X_train.columns)},
            os.path.join(OUT_DIR, "best_model.joblib"))
print(f"\nSaved best model ({best_name}) + threshold to {OUT_DIR}/best_model.joblib")
print(f"Plots saved to {PLOT_DIR}")
print("""
To predict on a new applicant:
    pre = joblib.load('output/preprocessor.joblib')
    bundle = joblib.load('output/best_model.joblib')
    # new_df must go through the SAME feature engineering as the preprocessing script
    proba = bundle['model'].predict_proba(pre.transform(new_df))[:, 1]
    decision = proba >= bundle['threshold']
""")


Saved best model (LogisticRegression) + threshold to output/best_model.joblib
Plots saved to output\model_plots

To predict on a new applicant:
    pre = joblib.load('output/preprocessor.joblib')
    bundle = joblib.load('output/best_model.joblib')
    # new_df must go through the SAME feature engineering as the preprocessing script
    proba = bundle['model'].predict_proba(pre.transform(new_df))[:, 1]
    decision = proba >= bundle['threshold']

